In [0]:
CREATE OR REPLACE TABLE IDENTIFIER(:catalog).bronze.region AS
SELECT *, current_timestamp() AS _ingested_at, _metadata.file_path AS _source_file FROM samples.tpch.region;

CREATE OR REPLACE TABLE IDENTIFIER(:catalog).bronze.nation AS
SELECT *, current_timestamp() AS _ingested_at, _metadata.file_path AS _source_file FROM samples.tpch.nation;

CREATE OR REPLACE TABLE IDENTIFIER(:catalog).bronze.customer AS
SELECT *, current_timestamp() AS _ingested_at, _metadata.file_path AS _source_file FROM samples.tpch.customer;

CREATE OR REPLACE TABLE IDENTIFIER(:catalog).bronze.supplier AS
SELECT *, current_timestamp() AS _ingested_at, _metadata.file_path AS _source_file FROM samples.tpch.supplier;

CREATE OR REPLACE TABLE IDENTIFIER(:catalog).bronze.part AS
SELECT *, current_timestamp() AS _ingested_at, _metadata.file_path AS _source_file FROM samples.tpch.part;

CREATE OR REPLACE TABLE IDENTIFIER(:catalog).bronze.partsupp AS
SELECT *, current_timestamp() AS _ingested_at, _metadata.file_path AS _source_file FROM samples.tpch.partsupp;

CREATE OR REPLACE TABLE IDENTIFIER(:catalog).bronze.orders AS
SELECT *, current_timestamp() AS _ingested_at, _metadata.file_path AS _source_file FROM samples.tpch.orders;

CREATE OR REPLACE TABLE IDENTIFIER(:catalog).bronze.lineitem AS
SELECT *, current_timestamp() AS _ingested_at, _metadata.file_path AS _source_file FROM samples.tpch.lineitem;

In [0]:
WITH source_counts AS (
  SELECT 'region' AS table_name, COUNT(*) AS row_count FROM samples.tpch.region
  UNION ALL SELECT 'nation', COUNT(*) FROM samples.tpch.nation
  UNION ALL SELECT 'customer', COUNT(*) FROM samples.tpch.customer
  UNION ALL SELECT 'supplier', COUNT(*) FROM samples.tpch.supplier
  UNION ALL SELECT 'part', COUNT(*) FROM samples.tpch.part
  UNION ALL SELECT 'partsupp', COUNT(*) FROM samples.tpch.partsupp
  UNION ALL SELECT 'orders', COUNT(*) FROM samples.tpch.orders
  UNION ALL SELECT 'lineitem', COUNT(*) FROM samples.tpch.lineitem
),
bronze_counts AS (
  SELECT 'region' AS table_name, COUNT(*) AS row_count FROM ${catalog}.bronze.region
  UNION ALL SELECT 'nation', COUNT(*) FROM ${catalog}.bronze.nation
  UNION ALL SELECT 'customer', COUNT(*) FROM ${catalog}.bronze.customer
  UNION ALL SELECT 'supplier', COUNT(*) FROM ${catalog}.bronze.supplier
  UNION ALL SELECT 'part', COUNT(*) FROM ${catalog}.bronze.part
  UNION ALL SELECT 'partsupp', COUNT(*) FROM ${catalog}.bronze.partsupp
  UNION ALL SELECT 'orders', COUNT(*) FROM ${catalog}.bronze.orders
  UNION ALL SELECT 'lineitem', COUNT(*) FROM ${catalog}.bronze.lineitem
)
SELECT
  s.table_name,
  s.row_count AS source_rows,
  b.row_count AS bronze_rows,
  CASE WHEN s.row_count = b.row_count THEN 'MATCH' ELSE 'MISMATCH' END AS status
FROM source_counts s
JOIN bronze_counts b ON s.table_name = b.table_name
ORDER BY s.table_name